# Health vs. Wealth: A Housing-First Strategy for Malaria Mitigation in Kenya
**Authors:** Patrick Duffy, Daniel D'Alessio, and Austin Baron

## Executive Summary
This notebook analyzes clinical data to determine the best strategy for reducing the impact of malaria within Kenya. This code focuses on supplementing current research on malaria prevention in Kenya by examinign the best method for both reducing the speed of infection and the overall disease volume and burden. Specifically in Kenya, some Western counties have rates of incidence of over 600 times higher than Eastern counties. These models explore how time-to-infection can be reduced for Eastern counties that would benefit from avoiding initial malaria onset (Cox Time-Varying Proportional Hazards Model), how overall malaria cases can be reduced for Western counties with the highest malaria rates (Negative Binomial Regression), and how wall type alone can be a determining factor to predict malaria incidence in counties across Kenya (Linear Regression).

### Core Findings:
1. **Speed of Infection (Cox Time-Varying Proportional Hazards Model):** Mud walls and open eaves significantly accelerate the time-to-infection, even when adjusting for dynamic environmental mosquito pressure, and are the primary indicators for countries to use overall, even compared to a constructed housing index. This is likely due to the confounding variable of wealth incorporated into housing structure.
2. **Disease Volume (Negative Binomial Regression):** Mud walls and open eaves are also the most predictive features of overall malaria cases. Conversely, interventions like Indoor Residual Spraying (IRS) and window screens are more effective at reducing the total case volume than initial onset.
3. **The Bottom Line (Linear Regression):** As a county's average housing structural risk increases (has a greater proportion of the population with mud houses and less with brick), the total burden of disease rises linearly. This shows that clinical interventions alone are fighting an uphill battle against poor housing.

### Notebook Navigation:
* **Sections 1 - 3:** Data Ingestion, Cleaning, and Exploratory Data Analysis (EDA)
* **Section 5:** Micro-Level Risk (Cox Time-Varying Proportional Hazards Model)
* **Section 6:** Overall Disease Burden (Negative Binomial Regression Model)
* **Section 4:** The Macro Context - Kenya Natioanl Analysis (Linear Regression)

# Section 1: Project Setup and Data Ingestion
In this section, we load the required libraries and datasets for analysis

## 1.1 Library Imports
We used several general data analytics libraries, as well as lifelines, statsmodels, and sklearn for models.

In [ ]:
# ==========================================
# 1.1 LIBRARY IMPORTS
# ==========================================

# Data analysis
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Modeling and statistics
from lifelines import CoxTimeVaryingFitter
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

# Notebook settings
%matplotlib inline
warnings.filterwarnings('ignore')
sns.set_style("whitegrid")

print("Environment successfully configured.")

## 1.2 Data Ingestion & Source Verification
We load the provided datasets on first and overall incidence from the Core and Buffer study zones, precipitation data, and the entomological source file. For external datasets, we loaded data on malaria incidence in Kenya by county and number of wall types for homes by county in Kenya to provide a macro-level comparison for our regional findings.

In [ ]:
# ==========================================
# 1.2 DATA INGESTION
# ==========================================

# Clinical data
core_first = pd.read_csv("../data/B_D1_first_time_infection.csv")
core_overall = pd.read_csv("../data/B_D1_overall_infection.csv")
buffer_first = pd.read_csv("../data/B_D3_first_time_infection.csv")
buffer_overall = pd.read_csv("../data/B_D3_overall_infection.csv")

# Precipitation and mosquito datasets
precip_overall = pd.read_csv("../data/B_overall_infection_precipitations.csv")
precip_first = pd.read_csv("../data/B_first_time_infection_precipitation.csv")
combined = pd.read_csv("../data/combined_final.csv")

# External datasets
kenya_wall_type = pd.read_csv("../data/Kenya Wall Type by County.csv")
kenya_malaria_incidence = pd.read_csv("../data/Malaria Incidence in Kenya by County.csv")

print("Data ingestion complete.")

# Section 2: Data Cleaning and Integration
In this section, we merge clinical, environmental, and longitudinal datasets. We utilize a multi-tiered imputation strategy for missing data, establish dummies, and execute feature engineering.

## 2.1 Clinical Dataset Consolidation
We begin by vertically stacking core and buffer sets to create both a master first incidence dataset and a master overall incidence dataset.

In [ ]:
# ==========================================
# 2.1 CONSOLIDATING CORE AND BUFFER DATA
# ==========================================
# Stack datasets vertically to create master first incidence and overall incidence files
first_master = pd.concat([core_first, buffer_first], ignore_index=True)
overall_master = pd.concat([core_overall, buffer_overall], ignore_index=True)

# Print length of each dataset
print(f"Master clinical files created.")
print(f"First Infection Records: {len(first_master)}")
print(f"Overall Recurrent Records: {len(overall_master)}")

## 2.2 Environmental Data Merging and Imputation
We merge the precipitation data (1 and 2-month lags) with our master datasets. We use a hierarchial imputation strategy to fill precipitation for all households, firstly by cluster and date average, then by cluster average, then lastly by global median.

In [ ]:
# ==========================================
# 2.2 PRECIPITATION MERGE AND IMPUTATION
# ==========================================

# Desired columns in precipitation dataset
rain_cols = ['subject_id', 'date_of_followup', 'daily_precipitation_1month', 'daily_precipitation_2month']

# Merge rain data into both masters
first_master = pd.merge(first_master, precip_first[rain_cols], on=['subject_id', 'date_of_followup'], how='left')
overall_master = pd.merge(overall_master, precip_overall[rain_cols], on=['subject_id', 'date_of_followup'], how='left')

# First merge missing values by cluster and date, then by cluster average, then lastly by global median
for df in [first_master, overall_master]:
    for col in ['daily_precipitation_1month', 'daily_precipitation_2month']:
        # 1. By cluster and date average
        df[col] = df.groupby(['cluster', 'date_of_followup'])[col].transform(lambda x: x.fillna(x.mean()))
        # 2. By general cluster average
        df[col] = df.groupby('cluster')[col].transform(lambda x: x.fillna(x.mean()))
        # 3. By global median
        df[col] = df[col].fillna(df[col].median())

print("Precipitation merging and multi-level imputation complete.")

## 2.3 Entomological Exposure Mapping
Extract monthly mosquito counts per cluster from the "combined" dataset, and fill data for missing values first by average month and year, then by average cluster value, then lastly by the study average.

In [ ]:
# ==========================================
# STEP 2.3: MOSQUITO DATA INTEGRATION
# ==========================================

# Clean columns
combined.columns = combined.columns.str.strip()

# 1. Create a month-year key for matching

# Make a YYYY-MM version of the follow-up date
for df in [first_master, overall_master]:
    df['month_yr'] = pd.to_datetime(df['date_of_followup'], format='mixed', errors='coerce').dt.strftime('%Y-%m')

# Do the same for the combined dataset
combined['month_yr'] = pd.to_datetime(combined['date_of_followup'], format='mixed', errors='coerce').dt.strftime('%Y-%m')

# 2. Merge mosquito counts onto main dataset

# Average mosquito counts by cluster + month
mosq_lookup = combined.groupby(['cluster', 'month_yr'])['daily_anoph_count_per_hh_by_mont'].mean().reset_index()

# Left join to keep all patient entries
first_master = pd.merge(first_master, mosq_lookup, on=['cluster', 'month_yr'], how='left')
overall_master = pd.merge(overall_master, mosq_lookup, on=['cluster', 'month_yr'], how='left')

# 3. Fill in mosquito data by average month value across all clusters, then by cluster average, then by overall study average

target_col = 'daily_anoph_count_per_hh_by_mont'

for df in [first_master, overall_master]:
    # 1. By average month value across all clusters
    df[target_col] = df[target_col].fillna(df.groupby('month_yr')[target_col].transform('mean'))
    # 2. By cluster average over full study
    df[target_col] = df[target_col].fillna(df.groupby('cluster')[target_col].transform('mean'))
    # 3. By overall study average
    df[target_col] = df[target_col].fillna(df[target_col].mean())
    
    # Drop temporary key
    df.drop(columns=['month_yr'], inplace=True)

#Print missing mosquito data
print("Mosquito data merged")
print(f"Missing mosquito data in first_master: {first_master[target_col].isnull().sum()}")
print(f"Missing mosquito data in overall_master: {overall_master[target_col].isnull().sum()}")

## 2.4 Feature Engineering and Variable Encoding
We convert visit date entries into "Days Since Enrollment" to track patient participation duration. We also establishh dummy variables for analysis.

In [ ]:
# ==========================================
# 2.4 FEATURE ENGINEERING & ENCODING
# ==========================================

# Standardize the duration of study participation
for df in [first_master, overall_master]:
    df['date_of_followup'] = pd.to_datetime(df['date_of_followup'], errors='coerce')
    df['enrollment_date'] = pd.to_datetime(df['enrollment_date'], errors='coerce')
    df['days_since_enrollment'] = (df['date_of_followup'] - df['enrollment_date']).dt.days

# Convert text categories into dummy model-ready 1s and 0s
cat_cols = ['core_or_buffer', 'wall_type', 'gender', 'treatment_allocation', 'eaves_open', 'final_diagnosis', 'roof_type']

first_master_clean = pd.get_dummies(first_master, columns=[c for c in cat_cols if c in first_master.columns], drop_first=True).replace({True: 1, False: 0})
overall_master_clean = pd.get_dummies(overall_master, columns=[c for c in cat_cols if c in overall_master.columns], drop_first=True).replace({True: 1, False: 0})

# Rename columns to more clear names
name_map = {
    'daily_anoph_count_per_hh_by_mont': 'daily_anoph_count_per_hh_by_month',
    'core_or_buffer_Core': 'is_core', 
    'wall_type_mud': 'is_wall_mud', 
    'eaves_open_open': 'is_eaves_open', 
    'final_diagnosis_positive': 'is_diagnosis_positive',
    'treatment_allocation_SR': 'is_spray'
}

for df in [first_master_clean, overall_master_clean]:
    df.rename(columns=name_map, inplace=True)

print(f"Features Engineered: {len(first_master_clean.columns)} columns ready for modeling.")

## 2.5 Model Preperation (Time-To-Event Intervals)
For the Cox Time-Series Proportional Hazards Model, we convert the visit-based records to be analyzed and define a start and stop time for each entry to trakc patient risk over the duration of the study.

In [ ]:
# ==========================================
# STEP 2.5: Prepare for Survival Analysis
# ==========================================

# Choose the features used in the survival analysis
champion_features = [
    'subject_id', 'visit', 'days_since_enrollment', 'is_diagnosis_positive',
    'age_at_enrollment', 'is_spray', 'is_core', 'is_wall_mud', 'is_eaves_open',
    'daily_anoph_count_per_hh_by_month'
]

# Construct the survival dataframe
df_tv_final = first_master_clean[[c for c in champion_features if c in first_master_clean.columns]].copy()
df_tv_final['stop'] = df_tv_final['days_since_enrollment'].astype(float)
df_tv_final['start'] = df_tv_final.groupby('subject_id')['stop'].shift(1).fillna(df_tv_final['stop'] - 1.0)
df_model_final = df_tv_final[df_tv_final['stop'] > df_tv_final['start']].dropna()

print(f"Modeling dataset ready with column: {'daily_anoph_count_per_hh_by_month' in df_model_final.columns}")

## 2.6 Window Engineering
We calculate specific counts for unscreened windows (distinguishing between wood and glass) and identify open holes in the household structure (windows that are not glass, wooden, or screened) to account for relationships not present in the original dataset's window columns

In [ ]:
# ==========================================
# STEP: WINDOW FEATURE ENGINEERING
# ==========================================

# Calucluate more specific window types
for df in [first_master_clean, overall_master_clean]:
    # 1. Calculate unscreened wooden and glass windows
    unscreened_wooden = (df['number_wooden_windows'] - df['number_windows_screens']).clip(lower=0)
    unscreened_glass = (df['number_windows_glass'] - df['number_windows_screens']).clip(lower=0)

    # 2. Calculate wooden or glass with no screen, wooden with no screen, and glass with no screen
    # windows from previous values
    df['wood_glass_no_screen_windows'] = np.minimum(unscreened_wooden, unscreened_glass)
    df['wood_no_screen_windows'] = unscreened_wooden - df['wood_glass_no_screen_windows']
    df['glass_no_screen_windows'] = unscreened_glass - df['wood_glass_no_screen_windows']

    # 3. Identify open gaps originally counted as windows
    total_accounted = (df['number_windows_screens'] + df['wood_glass_no_screen_windows'] + 
                       df['wood_no_screen_windows'] + df['glass_no_screen_windows'])
    df['open_windows'] = (df['number_windows'] - total_accounted).clip(lower=0)

    # 4. Remove raw columns we are replacing with new measures
    df.drop(columns=['number_wooden_windows', 'number_windows_glass', 'number_windows'], inplace=True, errors='ignore')

print("Detailed window engineering complete.")

# Section 3: Exploratory Data Analysis (EDA)
In this section, we explore the relationships between housing structures and malaria prevalence. We analyze correlations, compare visit-level versus patient-level infection rates, and validate primary risk factors.

## 3.1 Correlation Heatmap (Multicolinearity)
This chart establishes the direction of risk and checks for multicollinearity between top variables to ensure no two variables are too related for the models to perform succesfully. A correlation greater than 0.7 would be a risk factor for our model.

In [ ]:
# ==========================================
# 3.1 CORRELATION HEATMAP
# ==========================================

# The correlation heatmap is used to test multicolinearity, where a value of
# over 0.7 would be concerning for future models.

plt.figure(figsize=(10, 8))

# Define variables in the correlation check
eda_cols = [
    'is_diagnosis_positive', 'age_at_enrollment', 'is_wall_mud', 
    'is_eaves_open', 'daily_anoph_count_per_hh_by_month',
    'daily_precipitation_1month', 'is_spray'
]

# Ensure only existing columns are used
valid_eda_cols = [c for c in eda_cols if c in overall_master_clean.columns]

# Calculate and display the matrix
corr_matrix = overall_master_clean[valid_eda_cols].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)

plt.title("Correlation Matrix: Baseline Risk Assessment", fontsize=14)
plt.tight_layout()
plt.show()

print("Takeaway: Mud walls and open eaves show a 0.38 correlation, and daily mosquito count and daily precipitation (1-month) shows a correlation of 0.35. This confirms these variables are related but distinct enough to be used as separate predictors.")

## 3.2 Target Distribution: Frequency vs. Prevelance
We compare infection rate by individual visit and by pateint to justify the use of complex longitudinal models.

In [ ]:
# ==========================================
# 3.2 INFECTION RATES: VISIT VS. PATIENT LEVEL
# ==========================================

# We compare the percentage of positive visits to positive patients.

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot the frequency of visits marked as positive for malaria throughout the study
sns.countplot(data=overall_master_clean, x='is_diagnosis_positive', ax=axes[0], palette='Set2')
axes[0].set_title("Infection Frequency (Per Visit)", fontsize=13)
axes[0].set_xlabel("Diagnosis (0 = Negative, 1 = Positive)")
axes[0].set_ylabel("Total Records")

# Plot the number of individal patients marked as having malaria throughout the study
patient_level = overall_master_clean.groupby('subject_id')['is_diagnosis_positive'].max().reset_index()
sns.countplot(data=patient_level, x='is_diagnosis_positive', ax=axes[1], palette='Set2')
axes[1].set_title("Infection Prevalence (Per Patient)", fontsize=13)
axes[1].set_xlabel("Ever Diagnosed Positive? (0 = No, 1 = Yes)")
axes[1].set_ylabel("Total Number of Patients")

plt.tight_layout()
plt.show()

print("Takeaway: While positive diagnoses are rare in individual visits, a significant portion of the patient population experiences at least one infection event.")

## 3.3 Probability of Infection by Wall Type and Eaves
We plot the probability of infection against mud walls and open eaves to see the relationship with incidence for the top two structural concerns.

In [ ]:
# ==========================================
# 3.3 STRUCTURAL RISK VISUALIZATION
# ==========================================

# We isolate top structural variables to verify their impact on infection probability before modeling.

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Chart the probability of malaria with (1) and without (0) a mud wall
sns.barplot(data=overall_master_clean, x='is_wall_mud', y='is_diagnosis_positive', ax=axes[0], palette='Reds')
axes[0].set_title("Infection Probability: Wall Type")
axes[0].set_xlabel("Wall Type (0 = Brick/Other, 1 = Mud)")
axes[0].set_ylabel("Infection Probability")

# Chart the probability of malaria with open eaves (1) and with closed eaves (0)
sns.barplot(data=overall_master_clean, x='is_eaves_open', y='is_diagnosis_positive', ax=axes[1], palette='Oranges')
axes[1].set_title("Infection Probability: Eaves Structure")
axes[1].set_xlabel("Eaves (0 = Closed, 1 = Open)")
axes[1].set_ylabel("Infection Probability")

plt.tight_layout()
plt.show()

print("Takeaway: Both mud walls and open eaves show a clear, visual increase in infection probability, validating our focus on structural interventions.")

# Section 4: Micro-Level Risk (Cox Time-Varying Proportional Hazards Model)
In this section, we use a Cox Time-Varying Proportional Hazards Model to quantify how variables accelerate/decelerate teh timeline of a patient's initial malaria infection.

## 4.1 Cox Model Assumption Checks
We verify the longitudinal data meeds the requirements for the model: all time intervals are positive and there are no missing values in the "Champion" feature set.

In [ ]:
# ==========================================
# 5.1 COX MODEL PRE-REQUISITES
# ==========================================

print("--- COX MODEL PRE-REQUISITES ---")

# Check that all stop values are less than or equal to start values
invalid_times = df_model_final[df_model_final['stop'] <= df_model_final['start']]
print(f"Negative/Zero time intervals: {len(invalid_times)} (Requirement: Must be 0)")

# Check for any missing data
missing_data = df_model_final.isnull().sum().sum()
print(f"Missing values in modeling dataset: {missing_data} (Requirement: Must be 0)")

print("\nTakeaway: Data meets the requirements for survival analysis. The time-varying architecture prevents Proportional Hazard bias.")

## 4.2 Champion Model Training & Hazard Ratios
We use our five variables to initialize and fit the Cox Time-Varying Fitter. The model output is Hazard Ratios, showing how factors accelerate or delay the onset of malaria for patients in the study.

In [ ]:
# ==========================================
# 5.2 COX MODEL TRAINING (CHAMPION FEATURES)
# ==========================================

# Isolate the desired features
champion_features = [
    'subject_id', 'start', 'stop', 'is_diagnosis_positive', 
    'age_at_enrollment', 'is_spray', 'daily_anoph_count_per_hh_by_month', 
    'is_wall_mud', 'is_eaves_open'
]

# Create the final dataset
modeling_df = df_model_final[champion_features].copy()

# Initialize and fit the champion model with a penalizer of 0.1
ctv = CoxTimeVaryingFitter(penalizer=0.1)
ctv.fit(
    modeling_df, 
    id_col='subject_id', 
    event_col='is_diagnosis_positive', 
    start_col='start', 
    stop_col='stop', 
    show_progress=True
)

print("\n--- HAZARD RATIOS (SPEED OF INFECTION) ---")
ctv.print_summary()

## 4.3 Evaluation of Precision (LLRT)
We perform a Log-Likelihood Ratio Test (LLRT) to prove the model's accuracy. This compares the model to a null model and proves the current 5-variable model is significantly more precise than the baseline chance.

In [ ]:
# ==========================================
# 5.3 TIME-VARYING PRECISION (LLRT)
# ==========================================

# Run likelihood ratio test against null model
llrt = ctv.log_likelihood_ratio_test()

print("--- MODEL PRECISION EVALUATION (LLRT) ---")
print(f"Test Statistic: {llrt.test_statistic:.2f}")
print(f"P-Value: {llrt.p_value:.10f}")

# Interpret significance
if llrt.p_value < 0.05:
    print("\nTakeaway: The p-value is near zero, meaning our model is significnatly more accurate at predicting infectinon speed than a null model.")
else:
    print("\nTakeaway: The model does not perform significantly better than a null model.")

## 4.4 Visualizing Structural Hazard Ratios
We visualize the positive or negative impact of each variable by using a Forest Plot, where features on the left decrease risk (slow down infection) and features on the right increase risk (accelerating infection).

In [ ]:
# 5.4 VISUALIZING THE HAZARD RATIOS
# ==========================================
import matplotlib.pyplot as plt

# Plot feature impact on forest plot
plt.figure(figsize=(7, 5))
ctv.plot()

plt.title("Hazard Ratios: Impact of Structural and Environmental Factors", fontsize=14)
plt.xlabel("Log Hazard Ratio ( > 0 means Higher Risk / Faster Infection )")
plt.axvline(0, color='red', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# Added: Print a clean table of Hazard Ratios for the presentation
print("--- EXACT HAZARD RATIOS (exp(coef)) ---")
print(ctv.summary['exp(coef)'].round(3))

print("\nTakeaway: Mud walls and open eaves clearly push to the right of the baseline, proving they significantly accelerate the timeline of a malaria infection.")

## 4.5 Evaluation of Bias: Multicollinearity (VIF)
We calculate the Variance Inflation Factor (VIF) to ensure the Hazard Ratios are not biased by overlapping variables, with score below 5 showing the possibly related structural and environmental factors are statistically independent.

In [ ]:
# ==========================================
# 5.5 CHAMPION MODEL BIAS CHECK (VIF)
# ==========================================

from statsmodels.stats.outliers_influence import variance_inflation_factor

# Isolate just our 5 independent variables
X_vif_champion = modeling_df[[
    'age_at_enrollment', 
    'is_spray', 
    'daily_anoph_count_per_hh_by_month', 
    'is_wall_mud', 
    'is_eaves_open'
]].dropna()

# 2. Calculate VIF for each variable
vif_champion = pd.DataFrame()
vif_champion["Feature"] = X_vif_champion.columns
vif_champion["VIF"] = [variance_inflation_factor(X_vif_champion.values, i) for i in range(X_vif_champion.shape[1])]

# 3. Display sorted results
print("--- VARIANCE INFLATION FACTOR (BIAS CHECK) ---")
print(vif_champion.sort_values(by="VIF", ascending=False).to_string(index=False))

print("\nTakeaway: All VIF scores are below 5, proving our Hazard Ratios are statistically stable and unbiased.")

# Section 5: Overall Disease Burden (Negative Binomial Regression Model)
In this section, we use a Negative Binomial regression to calculate how factors increase/decrease a pateint's cumulative annual malaria infections.

## 5.1 Target Distribution & Data Setup
We aggregate entries into single values for each patient, summing the total number of postiive diagnoses and their average enviornmental exposure.

In [ ]:
# ==========================================
# 5.1 TARGET DISTRIBUTION & DATA SETUP
# ==========================================

# Aggregate 'is_diagnosis_positive' to count total infections
# We group by subject_id to see the total burden over the duration of the study
df_nb = overall_master_clean.groupby('subject_id').agg({
    'is_diagnosis_positive': 'sum',
    'age_at_enrollment': 'first',
    'is_spray': 'max',
    'is_wall_mud': 'max',
    'is_eaves_open': 'max',
    'number_windows_screens': 'mean',
    'glass_no_screen_windows': 'mean'
}).dropna()

# Define features and target columns
X_nb = df_nb.drop(columns=['is_diagnosis_positive'])
y_nb = df_nb['is_diagnosis_positive']
X_nb_const = sm.add_constant(X_nb)

print("Patient-level count data successfully aggregated.")

## 5.2 Negative Binomial Model & IRR Visualization
We fit the Negative Binomial model and calculate the Incidence Rate Ratio for each factor, which calculates the multiplier effect a factor has on the total number of malaria cases for a patient.

In [ ]:
# ==========================================
# 5.2 CHAMPION MODEL & IRR VISUALIZATION
# ==========================================

# Fit the Negative Binomial Model
champion_nb = sm.NegativeBinomial(y_nb, X_nb_const).fit(disp=0)

# Calculate Incidence Rate Ratios (IRR)
irr_df = pd.DataFrame({
    'Feature': champion_nb.params.index,
    'IRR': np.exp(champion_nb.params.values),
    'P-Value': champion_nb.pvalues.values
})

# Filter out the constant and alpha for the graph, and sort from greatest to least by IRR
irr_df = irr_df[~irr_df['Feature'].isin(['const', 'alpha'])].sort_values(by='IRR', ascending=False)

# 3. Plot the IRRs for ecah feature
plt.figure(figsize=(10, 6))
sns.barplot(x='IRR', y='Feature', data=irr_df, palette='coolwarm_r')
plt.axvline(1.0, color='black', linestyle='--', linewidth=2) # The "Neutral" line

plt.title("Actionable Interventions: Impact on Total Malaria Volume", fontsize=14)
plt.xlabel("Incidence Rate Ratio ( < 1.0 Reduces Cases | > 1.0 Multiplies Cases )")
plt.tight_layout()
plt.show()



print("\nTakeaway: Mud walls and open eaves significantly multiply a patient's total malaria volume.")
print("The spray and unscreened glass windows have the greatest positive effect on reducing the overall impact of malaria in the study.")

## 5.3 Handling Overdispersion (Alpha Check)
We examine the alpha parameter to see whether the dat shows overdispersion. If alpha is significantly different from zero, the data is overdispersed and a Negative Binomial model is more statistically importnat than a standard Poisson model in handling this variance.

In [ ]:
# ==========================================
# 5.3 ASSUMPTION CHECK: OVERDISPERSION (ALPHA)
# ==========================================

# Calculate alpha and its associated p-value
alpha_value = champion_nb.params['alpha']
alpha_pvalue = champion_nb.pvalues['alpha']

print("--- ASSUMPTION CHECK: OVERDISPERSION ---")
print(f"Alpha Parameter: {alpha_value:.3f}")
print(f"Alpha P-Value: {alpha_pvalue:.5f}")

# Interpretate whether overdispersion is statistically relevant
print("\nTakeaway: The near-zero p-value for alpha provides definitive mathematical proof that the data is overdispersed.")
print("This confirms the Negative Binomial was the correct choice over a Poisson model.")

## 5.4 Evaluation of Bias: Multicollinearity (VIF)
We calculate the VIF for the final feature set to ensure feature independence and verify that IRR multipliers are statistically reliable.

In [ ]:
# ==========================================
# 5.4 MODEL BIAS: MULTICOLLINEARITY (VIF)
# ==========================================

# Calculate VIF for our final feature set
vif_final = pd.DataFrame()
vif_final["Feature"] = X_nb.columns
vif_final["VIF"] = [variance_inflation_factor(X_nb.values, i) for i in range(X_nb.shape[1])]

print("--- BIAS CHECK: MULTICOLLINEARITY (VIF) ---")
print(vif_final.sort_values(by="VIF", ascending=False).to_string(index=False))

# Evaluate VIF scores
print("\nTakeaway: All VIF scores are below the strict 5, confirming model features are unbiased.")

## 5.5 Evaluation of Precision & Error
We evaluate the precision fo the model by calculating teh Mean Absolute Error (MAE), which shows how many malaria cases, on average, that our model's predictions deviate from the report data records.

In [ ]:
# ==========================================
# 5.5 PRECISION & ERROR EVALUATION
# ==========================================

# Predict the counts based on the model
y_pred_nb = champion_nb.predict(X_nb_const)

# Calculate Mean Absolute Error (MAE)
mae_nb = mean_absolute_error(y_nb, y_pred_nb)

print("--- PRECISION CHECK: MEAN ABSOLUTE ERROR ---")
print(f"Mean Absolute Error (MAE): {mae_nb:.3f} cases")

# Intepret MAE value
print("\nTakeaway: The model's predictions deviate by only 1.179 case per patient, demonstrating high precision.")

## 5.6 Final Impact Analysis: Percent Change in Malaria Burden
We convert the IRRs into percent changes by feature to allow us to understand how much a specific feature affects total malaria cases and make our findings more usable.

In [ ]:
# ==========================================
# 5.6 FINAL IMPACT ANALYSIS (SIMPLE)
# ==========================================

# Calculate percent change from each IRR
irr_df['Percent_Impact'] = (irr_df['IRR'] - 1) * 100


# We plot the percent changes in order of negative impact
# Red is used to display harmful features, and green for preventative
plt.figure(figsize=(10, 6))
irr_df = irr_df.sort_values(by='Percent_Impact', ascending=False)
colors = ['#d9534f' if x > 0 else '#5cb85c' for x in irr_df['Percent_Impact']]

sns.barplot(x='Percent_Impact', y='Feature', data=irr_df, palette=colors)

plt.axvline(0, color='black', linewidth=1.5, alpha=0.7)
plt.title("Malaria Impact: How much do these factors change the case count?", fontsize=14)
plt.xlabel("Percent Change in Total Cases (%)")
plt.grid(axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

# Print values diretly for interpretation
print("--- SUMMARY OF FINDINGS ---")
for i, row in irr_df.iterrows():
    direction = "Increase" if row['Percent_Impact'] > 0 else "Reduction"
    print(f"{row['Feature']:25} | {abs(row['Percent_Impact']):.1f}% {direction}")

# Section 6: The Macro Context - Kenya Natioanl Analysis (Linear Regression)
In this section, we use a linear regression to analyze whether the top feature in both models (wall type) is a predictor of malaria on a national scale in Kenya. We use external datasets on Kenya's incidence by county and total number of each wall type by county to test if a nation-wide realtionship exists outisde our clinical study (which only spanned Busia county) to inform a sustainable, permanent housing-first mitigation strategy.

## 6.1 Macro Data Integration & Normalization
We merge the two datasets and normalize the incidence data by using a log transformation to prepare it for linear regression analysis.

In [ ]:
# ==========================================
# STEP 1: CONSOLIDATING NATIONAL DATASETS
# ==========================================

# Merge malaria incidence onto wall-type data by county name (left join)
kenya_merged_data = pd.merge(kenya_wall_type, kenya_malaria_incidence, on='County Name', how='left')

# Calculate the proportion of households built with mud and brick
kenya_merged_data['General Mud'] = kenya_merged_data['Mud/Wood'] + kenya_merged_data['Mud/Cement']
kenya_merged_data['Proportion Mud'] = kenya_merged_data['General Mud'] / kenya_merged_data['Total households']
kenya_merged_data['Proportion Brick'] = kenya_merged_data['Brick/Block'] / kenya_merged_data['Total households']

# Apply log transformation to incidence rate values
kenya_merged_data['Log Incidence Rate'] = np.log1p(kenya_merged_data['Incidence Rate'])

# Display number of counties captured in data
print(f"Data merge complete. Analyzing structural patterns across {len(kenya_merged_data)} Kenyan counties.")

## 6.2 National Regional Trends
We visualize the raw incidence rates by county before modeling to identify variation in malaria across regions in Kenya.

In [ ]:
# ==========================================
# STEP 2: VISUALIZING NATIONAL TRENDS
# ==========================================

# Plot incidence trend by county
plt.figure(figsize=(12, 8))
sns.barplot(x='County Name', y='Incidence Rate', 
            data=kenya_malaria_incidence.sort_values(by='Incidence Rate', ascending=False),
            palette='viridis')

plt.xticks(rotation=90)
plt.title('Malaria Incidence Rates in Kenya by County', fontsize=14)
plt.ylabel('Incidence Rate (Cases per 1,000)')
plt.tight_layout()
plt.show()


print("Takeaway: Significant regional inequality suggests that localized factors (such as housing) may predict these outcomes.")

## 6.3 Training the National Regression Model
We utilize a linear regression model to quantify how much of the national variance in malaria can be explained by the purely the proportion of having a mude wall instead of a brick one, since this was the most predictive feature in both models.

In [ ]:
# ==========================================
# STEP 3: NATIONAL REGRESSION TRAINING
# ==========================================

# Remove non-finite values that break regression math
clean_macro = kenya_merged_data.replace([np.inf, -np.inf], np.nan).dropna(
    subset=['Log Incidence Rate', 'Proportion Mud', 'Proportion Brick']
)

# Define X (structural features) and y (malaria burden)
X_log = clean_macro[['Proportion Mud', 'Proportion Brick']]
y_log = clean_macro['Log Incidence Rate']

# Split 80/20 for training and validation
X_train_log, X_test_log, y_train_log, y_test_log = train_test_split(
    X_log, y_log, test_size=0.2, random_state=42
)

# Train the model
model_log = LinearRegression()
model_log.fit(X_train_log, y_train_log)
y_pred_log = model_log.predict(X_test_log)

print("Macro Linear Regression successfully trained on national data.")

## 6.4 Evaluating Model Bias & Residuals
To ensure our macro conclusions are statistically valid, we evaluate the model's residuals for randomness and normality.

### 6.4.1 Residuals vs. Fitted (Linearity and Homoscedasticity)
We check for homoscedasticity by plotting the residuals versus fitted values, where a random scatter of points without a pattern is consitent across all predicted levels of malaria.

In [ ]:
# ==========================================
# STEP 4A: RESIDUAL VS. FITTED PLOT
# ==========================================

# Plot the residual values versus the fitted ones

residuals_log = y_test_log - y_pred_log

plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_pred_log, y=residuals_log)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Fitted Values (Log Incidence Rate)')
plt.ylabel('Residuals')
plt.title('Residuals vs. Fitted Values (Homoscedasticity Check)')
plt.grid(True)
plt.show()
print("Takeaway: Residuals are scattered around the zero line with a slight upward pattern as fitted values increase. This suggests a small deviation from perfect homoscedasticity, but no strong pattern that would reject the regression assumptions.")

### 6.4.2 Q-Q Plot of Residuals (Normality)
We compare the distribution of the model's errors against a theoretical normal distribution using a Quantile-Quantile (Q-Q) plot. If the points roughly fall along the diagnol line, the assumption of normally distributed errors is satisfied.

In [ ]:
# ==========================================
# STEP 4B: Q-Q PLOT OF RESIDUALLS
# ==========================================

# Plot the Quantile-Quantile plot to examine if errors are normally distributed

plt.figure(figsize=(10, 6))
sm.qqplot(residuals_log, line='s')
plt.title('Q-Q Plot of Residuals (Normality Check)')
plt.grid(True)
plt.show()

print("Takeaway: Points following the diagonal line indicate that the model's errors are normally distributed.")

### 6.4.3 Histogram of Residuals (Normality)
For the final validation, we visualize the frequency of residuals on a histogram. A bell-shaped curve centered at zero confirms the model's predictions are accurate without skewing in a specific direction.

In [ ]:
# ==========================================
# STEP 4C: HISTOGRAM OF RESIDUALS
# ==========================================

# Plot histogram of residualsto see if prediction accuracy skews in a certain direction.

plt.figure(figsize=(10, 6))
sns.histplot(residuals_log, kde=True, color='teal')
plt.xlabel('Residual Error Value')
plt.ylabel('Frequency')
plt.title('Histogram of Residuals (Distribution Check)')
plt.grid(True)
plt.show()

print("Takeaway: The residuals are roughly bell-shaped but show a slight shift away from zero, indicating a small tendency for the model to over- or under-predict in some cases. However, the distribution is still roughly normal.")

### 6.4.4 Cook's Distance (Influential Points)

In [ ]:
# ==========================================
# STEP 4D: COOK'S DISTANCE (INFLUENTIAL POINT CHECK)
# ==========================================

# Refit regression using statsmodels for diagnostics
X_train_sm = sm.add_constant(X_train_log)

sm_model = sm.OLS(y_train_log, X_train_sm).fit()

# Calculate Cook's Distance
influence = sm_model.get_influence()
cooks = influence.cooks_distance[0]

plt.figure(figsize=(10,6))
plt.stem(cooks)
plt.title("Cook's Distance for Each Observation")
plt.xlabel("Observation Index")
plt.ylabel("Cook's Distance")
plt.show()

threshold = 4 / len(X_train_log)
print(f"Cook's Distance threshold: {threshold:0.2}")
print("\nTakeaway: One observation slightly exceeds the Cook’s Distance screening threshold of 0.11, suggesting higher influence. However, its value remains relatively small, indicating that no single observation significantly influencies the regression results.")

## 6.5 Macro Model Interpretation
Now that assumptions are validated, we interpret the final coefficients and R-squared values to quantify the impact of mud or brick walls on malaria incidence across Kenya

In [ ]:
# ==========================================
# STEP 5: FINAL NATIONAL RESULTS
# ==========================================
r2_score = model_log.score(X_test_log, y_test_log)

print("--- KENYA NATIONAL ANALYSIS RESULTS ---")
print(f"R-Squared (Explanatory Power): {r2_score:.4f}")
print(f"Coefficient for Mud Housing:   {model_log.coef_[0]:.4f}")
print(f"Coefficient for Brick Housing: {model_log.coef_[1]:.4f}")
print(f"Model Intercept:               {model_log.intercept_:.4f}")

print("\nFinal Takeaway: The positive relationship between mud housing and malaria incidence is statistically significant at the national level, justifying our move to micro-level patient modeling.")